# 🏥 Patient Recovery Prediction using Multiple Regression
### Supervised Machine Learning — Multi-Output Regression
---

## 📌 Problem Statement

A hospital wants to predict **two recovery outcomes** for patients based on their health profile and treatment plan:

| Target Variable | Description |
|---|---|
| `Recovery_Days` | Number of days until the patient is discharged |
| `Pain_Reduction_Score` | Score (0–100) indicating how much pain was reduced |

**Input Features:** Age, BMI, Blood Pressure, Therapy Hours, Medication Dosage, Pre-existing Conditions Score

**Why Multiple Regression with Two Targets?**
- Standard linear regression predicts **one** target at a time.
- `MultiOutputRegressor` wraps a base model to **independently predict each target** simultaneously.
- This is more efficient and convenient than running two separate models.

---

## Step 1 — Import Libraries

> We use `MultiOutputRegressor` from `sklearn.multioutput` — it wraps a base estimator
> and fits one regressor **per target column** automatically.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✅ Libraries imported successfully!')

✅ Libraries imported successfully!


## Step 2 — Load Dataset

> The dataset contains **120 patient records** with 6 features and **2 target variables**.
> A few missing values exist in `Blood_Pressure` and `Medication_Dosage` — we will handle them in Step 3.


In [2]:
df = pd.read_csv('patient_recovery_data.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (120, 8)


,Age,BMI,Blood_Pressure,Therapy_Hours,Medication_Dosage,Preexisting_Score,Recovery_Days,Pain_Reduction_Score
0,58,35.5,129.0,14.3,76.3,0,67.2,0.0
1,71,19.2,71.0,16.1,76.7,4,44.7,7.7
2,48,39.7,70.0,17.9,14.8,9,77.4,0.0
3,34,34.9,117.0,7.4,90.7,8,98.7,0.0
4,62,22.0,81.0,8.1,53.0,5,64.6,0.0


## Step 3 — Exploratory Data Analysis (EDA)

> EDA helps us understand the data distribution, spot missing values, and see correlations.


In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check missing values
print('Missing values per column:')
print(df.isnull().sum())

### 3a — Fill Missing Values

> We fill missing values using the **column mean** — a simple and effective strategy for numerical features.
> ⚠️ Important: we fill the **entire dataframe** here before splitting (a simpler approach for beginners).
> In Step 6, we will show the production-safe method using only training data means.


In [ ]:
df.fillna(df.mean(numeric_only=True), inplace=True)
print('✅ Missing values filled with column mean.')
print('Remaining missing values:', df.isnull().sum().sum())

### 3b — Distribution of Target Variables

> Visualising the two target variables helps us understand if they are normally distributed
> or skewed — which affects model performance and interpretation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Recovery_Days'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Recovery Days')
axes[0].set_xlabel('Recovery Days')
axes[0].set_ylabel('Frequency')

axes[1].hist(df['Pain_Reduction_Score'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Distribution of Pain Reduction Score')
axes[1].set_xlabel('Pain Reduction Score (0–100)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 3c — Correlation Heatmap

> The heatmap shows how strongly each feature correlates with the targets.
> Values close to **+1 or -1** indicate strong relationships.


In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## Step 4 — Define Features & Targets

> - **X** = input features (the predictors the model learns from)
> - **y** = output targets — here we have **two columns** (multi-output)
>
> Unlike single-output regression where `y` is a 1-D array, here `y` is a **2-D DataFrame**
> with shape `(n_samples, 2)`.


In [3]:
features = ['Age', 'BMI', 'Blood_Pressure', 'Therapy_Hours',
            'Medication_Dosage', 'Preexisting_Score']

targets  = ['Recovery_Days', 'Pain_Reduction_Score']

X = df[features]
y = df[targets]

print(f'Features : {features}')
print(f'Targets  : {targets}')
print(f'X shape  : {X.shape}')
print(f'y shape  : {y.shape}   ← 2 columns (multi-output)')

Features : ['Age', 'BMI', 'Blood_Pressure', 'Therapy_Hours', 'Medication_Dosage', 'Preexisting_Score']
Targets  : ['Recovery_Days', 'Pain_Reduction_Score']
X shape  : (120, 6)
y shape  : (120, 2)   ← 2 columns (multi-output)


## Step 5 — Train / Test Split

> We reserve **20%** of data for testing and train on the remaining **80%**.
> `random_state=42` makes the split reproducible.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

Training samples : 96
Testing samples  : 24


## Step 6 — Handle Missing Values in Test Data (Safe Method)

> ⚠️ **Data Leakage Warning**
>
> When filling missing values, always use statistics computed **only from training data**.
> Using test-set statistics to fill missing values 'leaks' future information into training.
> Here we compute the mean from `X_train` and apply it to both sets.


In [5]:
train_means = X_train.mean()

X_train = X_train.fillna(train_means)
X_test  = X_test.fillna(train_means)

print('✅ Missing values filled using training set mean only.')
print('Train missing:', X_train.isnull().sum().sum())
print('Test missing :', X_test.isnull().sum().sum())

✅ Missing values filled using training set mean only.
Train missing: 0
Test missing : 0


## Step 7 — Feature Scaling

> Features have very different ranges (e.g., Age: 20–80 vs Medication_Dosage: 5–100).
> `StandardScaler` transforms each feature to **mean=0, std=1**, so no single feature
> dominates due to its scale.
>
> ⚠️ We **fit** the scaler **only on training data** and apply (`transform`) to both sets.


In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on train, transform train
X_test  = scaler.transform(X_test)         # only transform test (no fit)

print('✅ Features scaled using StandardScaler (mean=0, std=1).')

## Step 8 — Build & Train the Multi-Output Model

> `MultiOutputRegressor(LinearRegression())` trains **one Linear Regression model per target**.
>
> Internally it fits:
> - Model A → predicts `Recovery_Days`
> - Model B → predicts `Pain_Reduction_Score`
>
> Both models share the same input features `X_train`.


In [6]:
base_model = LinearRegression()
model = MultiOutputRegressor(base_model)
model.fit(X_train, y_train)

print('✅ Multi-Output model trained!')
print(f'\nNumber of sub-estimators: {len(model.estimators_)}')
print(f'Sub-estimator type      : {type(model.estimators_[0]).__name__}')

# Show intercepts and coefficients for each target
for i, target in enumerate(targets):
    est = model.estimators_[i]
    print(f'\n--- {target} ---')
    print(f'  Intercept: {est.intercept_:.4f}')
    coef_df = pd.DataFrame({'Feature': features, 'Coefficient': est.coef_})
    print(coef_df.to_string(index=False))

✅ Multi-Output model trained!

Number of sub-estimators: 2
Sub-estimator type      : LinearRegression

--- Recovery_Days ---
  Intercept: -1.8208
          Feature  Coefficient
              Age     0.424106
              BMI     1.684658
   Blood_Pressure     0.085116
    Therapy_Hours    -2.044878
Medication_Dosage     0.059223
Preexisting_Score     2.409280

--- Pain_Reduction_Score ---
  Intercept: 6.7895
          Feature  Coefficient
              Age    -0.049740
              BMI    -0.145692
   Blood_Pressure    -0.018720
    Therapy_Hours     0.432908
Medication_Dosage     0.025047
Preexisting_Score    -0.486871


## Step 9 — Evaluate the Model

> We evaluate **each target separately** using three metrics:
>
> | Metric | Formula | What it means |
> |---|---|---|
> | **MAE** | avg(\|actual − predicted\|) | Average error in original units |
> | **RMSE** | √avg((actual − predicted)²) | Penalises large errors more |
> | **R²** | 1 − SS_res/SS_tot | % of variance explained (1.0 = perfect) |



### Evaluation Metric Formulas

**Mean Absolute Error (MAE):**
$$ \text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i| $$

**Root Mean Squared Error (RMSE):**
$$ \text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2} $$

**R² (Coefficient of Determination):**
$$ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $$

Where $y_i$ = actual, $\hat{y}_i$ = predicted, $\bar{y}$ = mean of actuals, $n$ = test samples


In [7]:
y_pred = model.predict(X_test)

print('=' * 50)
for i, target in enumerate(targets):
    actual    = y_test.iloc[:, i]
    predicted = y_pred[:, i]

    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2   = r2_score(actual, predicted)

    print(f'\n🎯 Target: {target}')
    print(f'   MAE  : {mae:.2f}')
    print(f'   RMSE : {rmse:.2f}')
    print(f'   R²   : {r2:.4f}  ({r2*100:.1f}% variance explained)')
print('=' * 50)


🎯 Target: Recovery_Days
   MAE  : 4.45
   RMSE : 5.93
   R²   : 0.9008  (90.1% variance explained)

🎯 Target: Pain_Reduction_Score
   MAE  : 2.38
   RMSE : 2.82
   R²   : 0.3363  (33.6% variance explained)


## Step 10 — Visualise Actual vs Predicted

> A scatter plot of **Actual vs Predicted** values is a quick sanity check.
> Points close to the diagonal line (y = x) indicate good predictions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

titles  = ['Recovery Days', 'Pain Reduction Score']
colors  = ['steelblue', 'salmon']

for i, (ax, title, color) in enumerate(zip(axes, titles, colors)):
    actual    = y_test.iloc[:, i]
    predicted = y_pred[:, i]

    ax.scatter(actual, predicted, alpha=0.7, color=color, edgecolors='white')
    lims = [min(actual.min(), predicted.min()) - 2,
            max(actual.max(), predicted.max()) + 2]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Perfect Prediction')
    ax.set_title(f'Actual vs Predicted — {title}')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.legend()

plt.tight_layout()
plt.show()

## Step 11 — Predict on New Patient Data

> Now we use the trained model to predict outcomes for **two new patients**.
> The new data must be scaled using the **same scaler fitted on training data**.


In [10]:
new_patients = pd.DataFrame({
    'Age'               : [45,  72 ],
    'BMI'               : [24.5, 31.2],
    'Blood_Pressure'    : [120, 145 ],
    'Therapy_Hours'     : [12,   5  ],
    'Medication_Dosage' : [50,  80  ],
    'Preexisting_Score' : [2,    7  ]
})

# Fill any missing (none here, but good practice) then scale
new_patients_filled = new_patients.fillna(train_means)

preds = model.predict(new_patients_filled)

new_patients['Predicted_Recovery_Days']      = preds[:, 0].round(1)
new_patients['Predicted_Pain_Reduction']     = preds[:, 1].round(1)

print('🏥 Predicted Patient Outcomes:')
print(new_patients.to_string(index=False))

🏥 Predicted Patient Outcomes:
 Age  BMI  Blood_Pressure  Therapy_Hours  Medication_Dosage  Preexisting_Score  Predicted_Recovery_Days  Predicted_Pain_Reduction
  45 24.5             120             12                 50                  2                     52.0                       4.2
  72 31.2             145              5                 80                  7                    105.0                      -3.3
